In [1]:
import importlib
from snowflake.snowpark.functions import col,trim,split,lit
from snowflake.snowpark.functions import col, sum as _sum, when, is_null
from snowflake.snowpark import functions as F
import sys
sys.path.append(r"C:\Users\G0004878\Desktop\TFT_Data\utils_files")
import snowflake_utils
import Snowflake_configuration
snowflake_conn_prop = Snowflake_configuration.ds1_role_json
from snowflake.snowpark.session import Session
import pandas as pd
import numpy as np
import datetime
from dateutil.relativedelta import relativedelta
from itertools import product

In [2]:
session = Session.builder.configs(snowflake_conn_prop).create()

In [3]:
session.use_database('MOP_DATABASE')
session.use_schema('SOQ')

### Read the festive table

In [4]:
df = session.table('MOP_DATABASE.SOQ.FESTIVE_DAYS_SOQ').to_pandas()

In [5]:
#Remove following festive columns
cols_to_drop = ['Navratri_days','Dussehra_(Vijayadashami)_days','Karwa_Chauth_days','Dhanteras_days','Diwali_days','Govardhan_Pooja_days','Bhai_Dooj_days','Chhath_Puja_days']
df.drop(columns=cols_to_drop,inplace=True)

In [6]:
upper_case_cols = [i.upper() for i in cols_to_drop]
upper_case_cols

['NAVRATRI_DAYS',
 'DUSSEHRA_(VIJAYADASHAMI)_DAYS',
 'KARWA_CHAUTH_DAYS',
 'DHANTERAS_DAYS',
 'DIWALI_DAYS',
 'GOVARDHAN_POOJA_DAYS',
 'BHAI_DOOJ_DAYS',
 'CHHATH_PUJA_DAYS']

In [7]:
#Create a dataframe

list_of_columns_part1 = [f"D-{i}" for i in range(30, 0, -1)]
list_of_columns_part2 = [f"D+{i}" if i!=0 else 'D0' for i in range(0,8)]

list_of_columns = list_of_columns_part1 + list_of_columns_part2

data = pd.DataFrame(columns=list_of_columns,index=range(2023,2027,1))

In [8]:
data.loc[2023,'D0'] = datetime.date(2023,11,12)
data.loc[2024,'D0'] = datetime.date(2024,10,31)
data.loc[2025,'D0'] = datetime.date(2025,10,20)
data.loc[2026,'D0'] = datetime.date(2026,11,8)

In [8]:
for k in product(list_of_columns,range(2023,2027,1)):
    if k[0]!='D0':
        diwali_date = data.loc[k[1],'D0']
        number = k[0].split('D')[1]
        data.loc[k[1],k[0]] = diwali_date + relativedelta(days=int(number))
        data.loc[k[1],k[0]] = data.loc[k[1],k[0]] + relativedelta(day=1)
    else:
        data.loc[k[1],k[0]] = data.loc[k[1],k[0]] + relativedelta(day=1)
        


In [9]:
data

,D-30,D-29,D-28,D-27,D-26,D-25,D-24,D-23,D-22,D-21,...,D-2,D-1,D0,D+1,D+2,D+3,D+4,D+5,D+6,D+7
2023,2023-10-01,2023-10-01,2023-10-01,2023-10-01,2023-10-01,2023-10-01,2023-10-01,2023-10-01,2023-10-01,2023-10-01,...,2023-11-01,2023-11-01,2023-11-01,2023-11-01,2023-11-01,2023-11-01,2023-11-01,2023-11-01,2023-11-01,2023-11-01
2024,2024-10-01,2024-10-01,2024-10-01,2024-10-01,2024-10-01,2024-10-01,2024-10-01,2024-10-01,2024-10-01,2024-10-01,...,2024-10-01,2024-10-01,2024-10-01,2024-10-01,2024-10-01,2024-10-01,2024-10-01,2024-10-01,2024-10-01,2024-10-01
2025,2025-09-01,2025-09-01,2025-09-01,2025-09-01,2025-09-01,2025-09-01,2025-09-01,2025-09-01,2025-09-01,2025-09-01,...,2025-10-01,2025-10-01,2025-10-01,2025-10-01,2025-10-01,2025-10-01,2025-10-01,2025-10-01,2025-10-01,2025-10-01
2026,2026-10-01,2026-10-01,2026-10-01,2026-10-01,2026-10-01,2026-10-01,2026-10-01,2026-10-01,2026-10-01,2026-10-01,...,2026-11-01,2026-11-01,2026-11-01,2026-11-01,2026-11-01,2026-11-01,2026-11-01,2026-11-01,2026-11-01,2026-11-01


In [10]:
#Create indices
indices = []
for i in range(2023,2027,1):
    k=list(data.loc[i,:].unique())
    for j in k:
        indices.append(j)
indices = sorted(list(set(indices)))

In [11]:
indices

[datetime.date(2023, 10, 1),
 datetime.date(2023, 11, 1),
 datetime.date(2024, 10, 1),
 datetime.date(2025, 9, 1),
 datetime.date(2025, 10, 1),
 datetime.date(2026, 10, 1),
 datetime.date(2026, 11, 1)]

In [12]:
# data.reset_index(drop=True,inplace=True)
# data.head()
data[:] = 1
data.head()

,D-30,D-29,D-28,D-27,D-26,D-25,D-24,D-23,D-22,D-21,...,D-2,D-1,D0,D+1,D+2,D+3,D+4,D+5,D+6,D+7
2023,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
2024,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
2025,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
2026,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1


In [20]:
new_df = pd.DataFrame(data=indices,columns=["MONTH_OF_SALE"])

In [21]:
new_df.shape

(7, 1)

In [22]:
new_df

,MONTH_OF_SALE
0,2023-10-01
1,2023-11-01
2,2024-10-01
3,2025-09-01
4,2025-10-01
5,2026-10-01
6,2026-11-01


In [23]:
data.drop_duplicates(inplace=True)

In [24]:
merged_df = pd.merge(left=new_df,right=data,how='cross')

In [25]:
merged_df.columns

Index(['MONTH_OF_SALE', 'D-30', 'D-29', 'D-28', 'D-27', 'D-26', 'D-25', 'D-24',
       'D-23', 'D-22', 'D-21', 'D-20', 'D-19', 'D-18', 'D-17', 'D-16', 'D-15',
       'D-14', 'D-13', 'D-12', 'D-11', 'D-10', 'D-9', 'D-8', 'D-7', 'D-6',
       'D-5', 'D-4', 'D-3', 'D-2', 'D-1', 'D0', 'D+1', 'D+2', 'D+3', 'D+4',
       'D+5', 'D+6', 'D+7'],
      dtype='object')

In [26]:
merged_df.shape

(7, 39)

In [28]:
merged_df.set_index('MONTH_OF_SALE',inplace=True)

In [29]:
merged_df

,D-30,D-29,D-28,D-27,D-26,D-25,D-24,D-23,D-22,D-21,...,D-2,D-1,D0,D+1,D+2,D+3,D+4,D+5,D+6,D+7
MONTH_OF_SALE,,,,,,,,,,,,,,,,,,,,,
2023-10-01,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
2023-11-01,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
2024-10-01,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
2025-09-01,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
2025-10-01,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
2026-10-01,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
2026-11-01,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1


In [30]:
merged_df.to_csv(r"Dataframe_with_D_plus_minus_features.csv")